# Configuracion Basica para usar Spark Sql

In [19]:
# Configurar Java y el entorno de Spark antes de importar PySpark

import os
import sys
from pathlib import Path

CONDA_PREFIX = Path(sys.prefix)

# Java instalado dentro del entorno de Anaconda
java_home = CONDA_PREFIX / "Library"

if not (java_home / "bin" / "java.exe").exists():
    raise FileNotFoundError(
        "No se encontró Java dentro del entorno de Anaconda.\n"
        "Abre Anaconda Prompt, activa el entorno spark y ejecuta:\n"
        "conda install -c conda-forge openjdk=8"
    )

os.environ["JAVA_HOME"] = str(java_home)
os.environ["PATH"] = str(java_home / "bin") + os.pathsep + os.environ.get("PATH", "")
os.environ.pop("SPARK_HOME", None)
os.environ["PYSPARK_PYTHON"] = sys.executable
os.environ["PYSPARK_DRIVER_PYTHON"] = sys.executable

#Hadoop
HADOOP_HOME = r"C:\hadoop"

os.environ["HADOOP_HOME"] = HADOOP_HOME
os.environ["PATH"] = (
    os.path.join(HADOOP_HOME, "bin")
    + os.pathsep
    + os.environ.get("PATH", "")
)

print("HADOOP_HOME:", os.environ["HADOOP_HOME"])
print("Hadoop bin:", os.path.join(HADOOP_HOME, "bin"))

print("Python:", sys.executable)
print("JAVA_HOME:", os.environ["JAVA_HOME"])

HADOOP_HOME: C:\hadoop
Hadoop bin: C:\hadoop\bin
Python: c:\Users\RyanHz\anaconda3\envs\spark\python.exe
JAVA_HOME: c:\Users\RyanHz\anaconda3\envs\spark\Library


In [21]:
# Crear la sesión de Spark

from pyspark.sql import SparkSession

spark = (
    SparkSession.builder
    .appName("HousingSparkSQL")
    .master("local[*]")
    .config("spark.driver.host", "127.0.0.1")
    .config("spark.driver.bindAddress", "127.0.0.1")
    .config("spark.ui.enabled", "false")
    .getOrCreate()
)

print("Spark iniciado correctamente")
print("Versión de Spark:", spark.version)

Spark iniciado correctamente
Versión de Spark: 3.5.6


In [22]:
# Localizar el archivo CSV

from pathlib import Path

posibles_archivos = [
    Path("../../../../Archivos-Analisis/files-tarea-m33/vgsales.csv")
]

ruta_csv = next((ruta for ruta in posibles_archivos if ruta.exists()), None)

if ruta_csv is None:
    raise FileNotFoundError(
        "No se encontró el CSV. Coloca 'vgsales.csv' "
        "en la misma carpeta que este notebook."
    )

print("Archivo encontrado:", ruta_csv.resolve())

Archivo encontrado: C:\Users\RyanHz\Documents\EBAC\VS\Archivos-Analisis\files-tarea-m33\vgsales.csv


# Contenido

In [28]:
vgsales_df = (
    spark.read
    .option("header", True)
    .option("inferSchema", True)
    .csv(str(ruta_csv))
)

In [29]:
vgsales_df.createOrReplaceTempView('VGSales')

sql_str = "select Publisher, sum(NA_Sales), sum(Global_Sales) from VGSales group by Genre, Publisher order by Publisher desc"
spark.sql(sql_str).show()

+--------------------+-------------------+------------------+
|           Publisher|      sum(NA_Sales)| sum(Global_Sales)|
+--------------------+-------------------+------------------+
|        responDESIGN|0.09000000000000001|              0.13|
|           mixi, Inc|                0.0|              0.86|
|inXile Entertainment|               0.02|               0.1|
|     imageepoch Inc.|                0.0|              0.01|
|     imageepoch Inc.|                0.0|              0.03|
|         id Software|               0.02|              0.03|
|                iWin|                0.0|              0.06|
|              fonfun|                0.0|              0.02|
|     dramatic create|                0.0|               0.1|
|     dramatic create|                0.0|              0.01|
|   bitComposer Games|                0.0|              0.03|
|   bitComposer Games|               0.16|              0.38|
|         Zushi Games|               0.04|              0.05|
|       

In [30]:
# Redondeo de las columnas
sql_str = "select Publisher, round(sum(NA_Sales), 2), round(sum(Global_Sales), 2) from VGSales group by Genre, Publisher order by Publisher desc"
spark.sql(sql_str).show()

+--------------------+-----------------------+---------------------------+
|           Publisher|round(sum(NA_Sales), 2)|round(sum(Global_Sales), 2)|
+--------------------+-----------------------+---------------------------+
|        responDESIGN|                   0.09|                       0.13|
|           mixi, Inc|                    0.0|                       0.86|
|inXile Entertainment|                   0.02|                        0.1|
|     imageepoch Inc.|                    0.0|                       0.01|
|     imageepoch Inc.|                    0.0|                       0.03|
|         id Software|                   0.02|                       0.03|
|                iWin|                    0.0|                       0.06|
|              fonfun|                    0.0|                       0.02|
|     dramatic create|                    0.0|                        0.1|
|     dramatic create|                    0.0|                       0.01|
|   bitComposer Games|   

In [31]:
spark.sql(sql_str).show(10, 40)
#         Numero de datos | numero de caracteres

+--------------------+-----------------------+---------------------------+
|           Publisher|round(sum(NA_Sales), 2)|round(sum(Global_Sales), 2)|
+--------------------+-----------------------+---------------------------+
|        responDESIGN|                   0.09|                       0.13|
|           mixi, Inc|                    0.0|                       0.86|
|inXile Entertainment|                   0.02|                        0.1|
|     imageepoch Inc.|                    0.0|                       0.01|
|     imageepoch Inc.|                    0.0|                       0.03|
|         id Software|                   0.02|                       0.03|
|                iWin|                    0.0|                       0.06|
|              fonfun|                    0.0|                       0.02|
|     dramatic create|                    0.0|                        0.1|
|     dramatic create|                    0.0|                       0.01|
+--------------------+---

In [32]:
# Se puede obtener el plan de ejecucion con explain

spark.sql(sql_str).explain()

# Plan extendido
# spark.sql(sql_str).explain(extended= True)


== Physical Plan ==
AdaptiveSparkPlan isFinalPlan=false
+- Sort [Publisher#720 DESC NULLS LAST], true, 0
   +- Exchange rangepartitioning(Publisher#720 DESC NULLS LAST, 200), ENSURE_REQUIREMENTS, [plan_id=623]
      +- HashAggregate(keys=[Genre#719, Publisher#720], functions=[sum(NA_Sales#721), sum(Global_Sales#725)])
         +- Exchange hashpartitioning(Genre#719, Publisher#720, 200), ENSURE_REQUIREMENTS, [plan_id=620]
            +- HashAggregate(keys=[Genre#719, Publisher#720], functions=[partial_sum(NA_Sales#721), partial_sum(Global_Sales#725)])
               +- FileScan csv [Genre#719,Publisher#720,NA_Sales#721,Global_Sales#725] Batched: false, DataFilters: [], Format: CSV, Location: InMemoryFileIndex(1 paths)[file:/c:/Users/RyanHz/Documents/EBAC/VS/Archivos-Analisis/files-tarea-..., PartitionFilters: [], PushedFilters: [], ReadSchema: struct<Genre:string,Publisher:string,NA_Sales:double,Global_Sales:double>




In [33]:
spark.sql(sql_str).columns

['Publisher', 'round(sum(NA_Sales), 2)', 'round(sum(Global_Sales), 2)']

# Particion de Datos

In [ ]:
# Implementacion de Partitionby
spark = SparkSession.builder.appName('Partitionby() PySpark').getOrCreate()

# Lee en el df el archivo
df = spark.read.option('header', True).csv('../../../../Archivos-Analisis/files-tarea-m33/vgsales.csv')

# Impriome el esquema
df.printSchema()

root
 |-- Rank: string (nullable = true)
 |-- Name: string (nullable = true)
 |-- Platform: string (nullable = true)
 |-- Year: string (nullable = true)
 |-- Genre: string (nullable = true)
 |-- Publisher: string (nullable = true)
 |-- NA_Sales: string (nullable = true)
 |-- EU_Sales: string (nullable = true)
 |-- JP_Sales: string (nullable = true)
 |-- Other_Sales: string (nullable = true)
 |-- Global_Sales: string (nullable = true)



## partition by
Generando pedazos de informacion

In [ ]:
df.write.option('header', True).partitionBy('Platform').mode('overwrite').csv('../../../../Archivos-Analisis/files-tarea-m33/partition/platform')

In [ ]:
df.write.option('header', True).partitionBy('Platform').mode('overwrite').csv('../../../../Archivos-Analisis/files-tarea-m33/partition/year')

## Coalesce y Repartition

In [38]:
# Implementacion de Partitionby
spark = SparkSession.builder.appName('coalesce() PySpark').getOrCreate()

# Lee en el df el archivo
df = spark.read.option('header', True).csv('../../../../Archivos-Analisis/files-tarea-m33/vgsales.csv')

# Impriome el esquema
df.printSchema()

root
 |-- Rank: string (nullable = true)
 |-- Name: string (nullable = true)
 |-- Platform: string (nullable = true)
 |-- Year: string (nullable = true)
 |-- Genre: string (nullable = true)
 |-- Publisher: string (nullable = true)
 |-- NA_Sales: string (nullable = true)
 |-- EU_Sales: string (nullable = true)
 |-- JP_Sales: string (nullable = true)
 |-- Other_Sales: string (nullable = true)
 |-- Global_Sales: string (nullable = true)



In [ ]:
# Genera en el directorio el numero de archivos solicitados
df.repartition(20).write.mode('overwrite').option('header', True).csv('../../../../Archivos-Analisis/files-tarea-m33/partition/rep')

In [40]:
# Usando coalesce
df2 = df.repartition(20)
df2.rdd.getNumPartitions()

20

In [41]:
df3 = df2.coalesce(10)

In [ ]:
df3.write.mode('overwrite').option('header', True).csv('../../../../Archivos-Analisis/files-tarea-m33/partition/coalesce')